In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd

import helpers

SEEDS = range(32, 37)
CONDITIONS = ['none', 'tabular', 'tabular-text', 'tabular-image', 'tabular-image-text']

gender_test = helpers.load_gender_test()
male = (gender_test == 0)
female = (gender_test == 1)

In [ ]:
class LateFusion(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, 40)
        self.fc2 = nn.Linear(40, 20)
        self.fc3 = nn.Linear(20, 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        return torch.sigmoid(self.fc3(x))

In [ ]:
rows = []

for condition in CONDITIONS:
    (labels_train_biased, labels_train_blind,
     labels_test_biased, labels_test_blind,
     tabular_train, tabular_test,
     text_train, text_test,
     faces_train, faces_test) = helpers.build_features(condition)

    modalities = [
        ('tabular', tabular_train, tabular_test),
        ('text', text_train, text_test),
        ('image', faces_train, faces_test),
    ]

    for seed in SEEDS:
        print(f"---SEED {seed}---")
        per_modality = {}

        for name, train, test in modalities:
            per_modality[name], _ = helpers.train_and_evaluate(
                lambda: LateFusion(train.shape[1]),
                [train],
                [test],
                labels_train_biased,
                seed,
            )

        predictions = np.mean(list(per_modality.values()), axis=0)
        parity, top100_female = helpers.computeDemographicParity(predictions, gender_test)
        equal_opportunity, n_selected = helpers.computeEqualityOfOpportunity(predictions, gender_test, labels_train_blind, labels_test_blind, verbose=(condition == 'none'), p = 75)

        rows.append({
            'condition': condition,
            'fusion': 'late',
            'seed': seed,
            'mae_biased': helpers.computeMAE(predictions, labels_test_biased),
            'mae_blind': helpers.computeMAE(predictions, labels_test_blind),
            'kl': helpers.computeKL(predictions[male], predictions[female]),
            'parity': parity,
            'top100_female': top100_female,
            'equal_opportunity': equal_opportunity,
            'n_selected': n_selected,
            'ndcg': helpers.computeNDCG(predictions, labels_test_blind),
            'mae_tabular_only': np.mean(np.abs(per_modality['tabular'] - labels_test_biased)),
            'mae_text_only': np.mean(np.abs(per_modality['text'] - labels_test_biased)),
            'mae_image_only': np.mean(np.abs(per_modality['image'] - labels_test_biased)),
        })

    print(condition, 'done')

results = pd.DataFrame(rows)
results.to_csv('../results/results_late.csv', index=False)
results.groupby('condition', sort=False)[['mae_biased', 'mae_blind', 'kl', 'parity', 'top100_female','equal_opportunity', 'n_selected', 'ndcg']].agg(['mean', 'std'])